# RESTOPS · Design a connected synthetic restaurant operation

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache/matplotlib"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from src.common import read, TABLEAU, REPORTS
from src.reporting import style
style()
pd.set_option("display.max_columns", 12)


**Business question:** What data does management need to diagnose restaurant performance?

We generate customer orders before product lines, link operational exports by stable identifiers, and introduce controlled export defects. No real company or customer data is used.

In [2]:
REGENERATE = False  # Set True only when deliberately rebuilding raw exports.
if REGENERATE:
    from src.generate_data import run
    run()
manifest = json.loads((ROOT / "data/raw/generation_manifest.json").read_text())
pd.Series({k: manifest[k] for k in ["seed", "orders_before_defects", "lines_before_defects"]})

seed                          42
orders_before_defects     796974
lines_before_defects     2111602
dtype: int64

In [3]:
pd.DataFrame({"dataset": list(manifest["raw_rows"]), "raw_rows": list(manifest["raw_rows"].values())})

,dataset,raw_rows
0,restaurants,18
1,products,24
2,transactions,2113902
3,labour,90738
4,customer_feedback,59822
5,loyalty,587968
6,promotions,108
7,waste,310447
8,targets,432
9,operating_costs,1296


In [4]:
read("restaurants").loc[:, ["restaurant_name", "state", "location_type", "seats"]]

,restaurant_name,state,location_type,seats
0,Sydney Central,NSW,CBD,90
1,Parramatta,NSW,Shopping centre,76
2,Bondi,NSW,Suburban,64
3,Newcastle,NSW,Regional,70
4,Chatswood,NSW,Shopping centre,82
5,Wollongong,NSW,Regional,65
6,Melbourne Central,VIC,CBD,96
7,Richmond,VIC,Suburban,68
8,Chadstone,VIC,Shopping centre,88
9,Geelong,VIC,Regional,66


### Inspect the source grain
A POS row is a product line, not an order. Null customer IDs are valid anonymous visits. Local timestamps require the restaurant timezone.

In [5]:
pd.read_csv(ROOT / "data/raw/transactions.csv", nrows=8)

,transaction_line_id,order_id,restaurant_id,restaurant_name,timestamp_local,product_id,...,promotion_id,list_revenue,discount_amount,net_revenue,gst_amount,ingredient_cost
0,1,1,1,Sydney Central,2024-01-01 14:35:16,10.0,...,NaN,22.0,0.0,22.0,2.2,7.1
1,2,1,1,Sydney Central,2024-01-01 14:35:16,3.0,...,NaN,21.0,0.0,21.0,2.1,6.3
2,3,2,1,Sydney Central,2024-01-01 14:06:16,12.0,...,NaN,8.0,0.0,8.0,0.8,1.8
3,4,2,1,Sydney Central,2024-01-01 14:06:16,16.0,...,NaN,5.0,0.0,5.0,0.5,1.2
4,5,3,1,Sydney Central,2024-01-01 20:10:50,22.0,...,NaN,12.0,0.0,12.0,1.2,3.7
5,6,3,1,Sydney Central,2024-01-01 20:10:50,9.0,...,NaN,17.0,0.0,17.0,1.7,4.9
6,7,4,1,Sydney Central,2024-01-01 19:33:31,14.0,...,NaN,7.0,0.0,7.0,0.7,1.5
7,8,4,1,Sydney Central,2024-01-01 19:33:31,8.0,...,NaN,25.0,0.0,25.0,2.5,9.2


In [6]:
pd.DataFrame([{k:v for k,v in defect.items() if k != "row_indices"} for defect in manifest["defects"]])

,table,column,type,injected_rows
0,transactions,restaurant_name,name_alias,2600
1,transactions,product_id,missing_key,800
2,transactions,timestamp_local,invalid_timestamp,650
3,transactions,quantity,quantity_outlier,350
4,transactions,net_revenue,invalid_revenue,250
5,products,category,category_alias,2
6,labour,hourly_cost,missing_rate,130
7,labour,paid_hours,invalid_hours,60
8,customer_feedback,overall_score,unanswered_survey,300
9,customer_feedback,overall_score,invalid_rating,50


**Decision:** Keep restaurant, product, order and date identifiers consistent across exports. Require both finance costs and operating context to explain profitability. The generation manifest is for independent QA; the cleaning module does not consume it.